# Perform a Layer by Layer forward pass on a model

This notebook is purely for debugging purposes. It is not meant to be run as a script. The goal is to perform a forward pass on a model layer by layer, and print the output of each layer, and compare it to the output of the RISC-V/C model.

TODO: Clean this up later, this is a mess right now.

In [1]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import random
import csv

from sklearn.model_selection import train_test_split

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

2025-05-08 15:06:48.565636: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-08 15:06:48.579620: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-08 15:06:48.695890: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-08 15:06:48.771760: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746698808.853260   28058 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746698808.87

In [4]:
# Load and preprocess MNIST dataset
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0  # Normalize the images to [0, 1]
x_train = np.expand_dims(x_train, -1)  # Add channel dimension
y_train = tf.keras.utils.to_categorical(y_train, 10)  # One-hot encode the labels

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 8s 1us/step


In [5]:
# Get a 1 random image
idx = random.randint(0, len(x_train) - 1)
image = x_train[idx].squeeze()  # (28, 28)
label = np.argmax(y_train[idx])  # Get label

with open("mnist_single_sample.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    
    # Flatten the image and combine it with the label
    row = [label] + image.flatten().tolist()
    
    # Write row to CSV
    writer.writerow(row)

print(label)

7


In [6]:
import numpy as np

def print_output_shape_and_values(x):
    print(f"Output shape: {x.shape}")
    
    # If it's a 4D tensor (e.g., batch of images), handle it
    if len(x.shape) == 4:
        _, height, width, channels = x.shape
        for c in range(channels):
            for i in range(height):
                for j in range(width):
                    print(f"{x[0, i, j, c]:.3f}", end=" ")
                print()
            print()
        
    
    # If it's a 2D array (after flattening), handle it
    elif len(x.shape) == 2:
        # Extract height and width from flattened shape, assume one image
        rows, cols = x.shape
        for i in range(rows):
            for j in range(cols):
                print(f"{x[i, j]:.3f}", end=" ")
            print()
    else:
        print("Unsupported shape")


## Load the model from mnist_cnn_model.keras

In [7]:
model = tf.keras.models.load_model("../models/mnist_cnn_model.keras")

2025-05-08 16:09:22.698173: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
x = tf.expand_dims(image, axis=0)  # Add batch dimension
x = tf.expand_dims(x, axis=-1)     # Add channel dimension

x = model.layers[0](x)  # First layer output

print_output_shape_and_values(x)

# Flatten the output and reshape it into 8 grids of 24x24
out = x.numpy().flatten()  # Flatten the output

# Reshape into 8 grids of 24x24
grids = out.reshape(8, 24, 24)

# Write to file with nine decimal places
with open("conv2d_out.txt", "w") as f:
    for i in range(24):  # Iterate over rows
        for grid in grids:  # Iterate over each grid
            row_values = ",".join(f"{grid[i, j]:.9f}" for j in range(24))  # Get the ith row of the grid with 9 decimal places
            f.write(row_values + ",")
        # Add a newline after each row from all grids
        # f.write(

Output shape: (1, 24, 24, 8)
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.386 -0.155 -0.129 -0.301 -0.379 -0.528 -0.525 -0.495 -0.478 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.365 -0.086 0.038 0.138 0.250 0.171 0.038 -0.108 -0.224 -0.209 -0.181 -0.182 -0.180 -0.192 -0.318 -0.521 -0.559 -0.529 -0.492 -0.460 -0.460 
-0.460 -0.460 -0.460 -0.466 -0.413 -0.264 0.044 0.404 0.532 0.502 0.438 0.381 0.354 0.320 0.316 0.350 0.373 0.326 0.121 -0.148 -0.3

In [9]:
# Second Layer
x = model.layers[1](x)  # Second layer output
print_output_shape_and_values(x)

Output shape: (1, 24, 24, 8)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.038 0.138 0.250 0.171 0.038 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.044 0.404 0.532 0.502 0.438 0.381 0.354 0.320 0.316 0.350 0.373 0.326 0.121 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.001 0.121 0.247 0.263 0.305 0.326 0.267 0.257 0.278 0.285

In [10]:
# Third Layer
x = model.layers[2](x)  # Third layer output
print_output_shape_and_values(x)

Output shape: (1, 12, 12, 8)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.038 0.404 0.532 0.438 0.354 0.350 0.373 0.121 0.000 0.000 
0.000 0.000 0.000 0.001 0.247 0.305 0.326 0.278 0.285 0.161 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.019 0.046 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.143 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.127 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.187 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.429 0.215 0.000 0.000 0.000 0.000 0.000 0.000 
0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.000 

0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 0.178 
0.178 0.281 0.803 1.0

In [11]:
# Fourth Layer
x = model.layers[3](x)  # Fourth layer output
print_output_shape_and_values(x)

Output shape: (1, 1152)
0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.281 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.803 0.000 0.000 0.131 0.088 0.289 0.000 0.000 1.053 0.000 0.000 0.440 0.000 0.157 0.000 0.000 1.063 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.480 0.000 0.067 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.000 0.000 0.000 0.000 0.000 0.000 0.000 0.178 0.00

In [12]:
# Fifth Layer: Dense Layer
x = model.layers[4](x)  # Fifth layer output
print_output_shape_and_values(x)

Output shape: (1, 10)
-11.932 -0.711 -9.981 -6.711 -15.833 -5.224 -26.694 10.943 0.420 -3.659 


In [13]:
# Sixth Layer: Softmax Layer
x = model.layers[5](x)  # Sixth layer output
print_output_shape_and_values(x)

Output shape: (1, 10)
0.000 0.000 0.000 0.000 0.000 0.000 0.000 1.000 0.000 0.000 


In [14]:
print(f"Predicted class: {np.argmax(x)}")
print(f"True class: {label}")

Predicted class: 7
True class: 7
